<h1>Chapter 8 - Multi-Agent Collaboration</h1>
<i>More Agents?!</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 8 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [1]:
from openai import OpenAI
from illustrated_agents.chapters.ch5_native import LLM

# Ollama through OpenAI API
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma4:e4b", client=client, think=True)

# Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = LLM(model="gemma-4-E4B-it-Q4_K_M", client=client)

# LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = LLM(model="gemma-4-e4b-it", client=client)

# Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = LLM(model="gemini-2.5-flash", client=client)

## 2. Agents as Tools

Agent orchestration can quickly be a daunting task, especially as you increasingly add many sub-Agents. There is a fortunately a nice trick to add collaboration amongst Agents using everything we have already created! The title gives it away, Agents as Tools. As covered in the book, there are various patterns that collaboration can take, but we focus on a central pattern where there is one Agent instructing a bunch of others. 

To run Agents as Tools, we first need to define them. So let's start by creating a `MathAgent`:

In [2]:
from illustrated_agents.chapters.ch5_native import NativeTools, LLM, Memory
from illustrated_agents.chapters.ch6_native import NativeReAct
from illustrated_agents.chapters.ch6_skills import Skills, TinyAgent
from illustrated_agents.chapters.ch6_skills import TinyAgent as ToolAgent


def add(a: str, b: str) -> float:
    return float(a) + float(b)

def subtract(a: str, b: str) -> float:
    return float(a) - float(b)

def multiply(a: str, b: str) -> float:
    return float(a) * float(b)

# Math specialist
math_tools = NativeTools()
math_tools.add_tool("add", add)
math_tools.add_tool("subtract", subtract)
math_tools.add_tool("multiply", multiply)

# Math Agent
math_agent = ToolAgent(
    llm=llm, 
    tools=math_tools, 
    memory=Memory(),
    skills=Skills(),
    planner=NativeReAct()
)

Note that we use the `TinyAgent` from chapter 5 which can only run a tool once and has no ReAct or Reflection. Next, we can convert the `math_agent` to a tool:

In [3]:
def ask_math_agent(question: str) -> str:
    """Delegate to the math specialist."""
    return math_agent.run(question)

## 3. The Orchestrator

We call the tool `ask_math_agent` so that your `TinyAgent` knows that the question is being rerouted to another agent. Now, let's create your `TinyAgent` with ReAct but without Reflection to make it more efficient:

In [4]:
# Tools - Add the Math Agent as a Tool!
tools = NativeTools()
tools.add_tool("ask_math_agent", ask_math_agent)

# Orchestrator Agent
orchestrator_agent = ToolAgent(
    llm=llm, 
    tools=tools, 
    memory=Memory(),
    skills=Skills(),
    planner=NativeReAct()
)

We named your `TinyAgent` the `orchestrator_agent` since it is in charge of using tools (including the `ask_math_agent` tool). We also use the same LLM for both Agents as that is easier since we do not have to run both of them on the samen device. In practice, however, you might want to use a smaller Agent as the `math_agent` since it only has to use tools. The `orchestrator_agent`, in contrast, tends to be a large model since it has to relay tasks and decide on the best action sequencing. 

Let's see if the `orchestrator_agent` you created works by asking it a basic question:

In [5]:
orchestrator_agent.run("What is 5.12 times 3.9?")

'5.12 times 3.9 is **19.968**.'

The output is correct, but let's see if your `TinyAgent` actually used the `math_agent`:

In [6]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(orchestrator_agent.trajectory)

It certainly did!

We can even explore the trajectory of the `math_agent`:

In [7]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(math_agent.trajectory)

Note that there is a "user" that asks "What is 5.12 times 3.9?". This is actually the `orchestrator_agent` that asked the question!

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how to create an orchestrator Agent that is capable of sending queries to sub-Agents. The "trick" that we used here is that we assigned the sub-Agent to be a tool which can be run as a regular tool would!

# What's Next

In the next chapter, we will explore how to go from your general purpose `TinyAgent` to a `CodingAgent`! It will require doing some work on the interface as we will building it to work in the terminal.